# AI-Assisted Form Capture for DHIS2Photograph of a paper health form → structured fields → DHIS2 tracker record,**verified by reading the record back out of DHIS2**.Run the cells in order. About four minutes.### SetupAdd your key to Colab Secrets (key icon, left sidebar) named exactly`ANTHROPIC_API_KEY`, with **Notebook access** switched ON. The underscoresmatter, and a missing toggle raises `SecretNotFoundError` even when the secretexists.### SafetyThe form is synthetic and the server is the public DHIS2 demo. Do not put realpatient data through this — images are sent to an API outside most implementingcountries, which is a legal question before it is a technical one.

In [ ]:
!pip install -q anthropicprint("ready")

## 1. ConnectionTwo things here are corrections of earlier mistakes worth knowing about.`play.dhis2.org/40` is retired. It returns an **nginx 404 HTML page**, not JSON,so blind parsing produces `Expecting value: line 1 column 1` and sends youhunting for a payload bug that does not exist. The check below reports the realcause.A pinned release is used rather than `/dev`. The dev instance tracks a movingSNAPSHOT build and resets nightly, which makes failures hard to attribute.

In [ ]:
import base64, json, re, time, uuid, requestsfrom datetime import date, datetimefrom google.colab import userdataDHIS2   = "https://play.im.dhis2.org/stable-2-40-12"AUTH    = ("admin", "district")          # public demo credentialsTIMEOUT = 30s = requests.Session()s.auth = AUTHs.headers.update({"Content-Type": "application/json", "Accept": "application/json"})r = s.get(f"{DHIS2}/api/system/info", timeout=TIMEOUT)try:    info = r.json()except ValueError:    raise SystemExit(        f"HTTP {r.status_code} and the body is not JSON. The server URL is "        f"probably wrong — retired DHIS2 hosts serve an nginx page. "        f"Body starts: {r.text[:120]}")print(f"connected : {DHIS2}")print(f"version   : {info.get('version')}")print(f"user      : {s.get(f'{DHIS2}/api/me', timeout=TIMEOUT).json().get('username')}")

## 2. ModelModel IDs change and a stale one fails with an unhelpful error. Ask the API whatexists rather than hardcoding a guess.

In [ ]:
import anthropicclient = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))available = [m.id for m in client.models.list(limit=30)]MODEL = next((m for m in available if "sonnet" in m), available[0])print("available :", ", ".join(available[:6]), "...")print("using     :", MODEL)

## 3. Discover the DHIS2 metadata**Never hardcode UIDs.** They are specific to one database, and DHIS2 rejectsunknown UIDs with messages that do not name the offending field. Copying themfrom another notebook is the most common cause of silent failure.Note the `generated` flag. Some mandatory attributes — Unique ID on thisprogramme — are created by the server and must **not** be supplied.

In [ ]:
PROGRAM_UID   = "IpHINAT79UW"      # Child ProgrammeORG_UNIT_NAME = "Ngelehun CHC"ou = s.get(f"{DHIS2}/api/organisationUnits",           params={"filter": f"name:eq:{ORG_UNIT_NAME}", "fields": "id,name"},           timeout=TIMEOUT).json()["organisationUnits"]assert ou, f"no organisation unit named {ORG_UNIT_NAME!r}"ORG_UNIT = ou[0]["id"]prog = s.get(f"{DHIS2}/api/programs/{PROGRAM_UID}",             params={"fields": "id,name,trackedEntityType[id,name],"                               "programTrackedEntityAttributes[mandatory,"                               "trackedEntityAttribute[id,name,valueType,generated]]"},             timeout=TIMEOUT).json()PROGRAM = prog["id"]TE_TYPE = prog["trackedEntityType"]["id"]ATTRS = [(p["trackedEntityAttribute"]["id"],          p["trackedEntityAttribute"]["name"],          p["trackedEntityAttribute"].get("valueType", ""),          bool(p.get("mandatory")),          bool(p["trackedEntityAttribute"].get("generated")))         for p in prog["programTrackedEntityAttributes"]]print(f"org unit  : {ou[0]['name']} = {ORG_UNIT}")print(f"programme : {prog['name']} = {PROGRAM}")print(f"TE type   : {prog['trackedEntityType']['name']} = {TE_TYPE}\n")print(f"{'UID':<14}{'name':<32}{'type':<12}flags")for uid, name, vt, mand, gen in ATTRS:    flags = " ".join(["REQUIRED"] if mand else []) + (" auto-generated" if gen else "")    print(f"{uid:<14}{name:<32}{vt:<12}{flags}")YOU_MUST_SUPPLY = [u for u, n, v, m, g in ATTRS if m and not g]print(f"\nyou must supply: {YOU_MUST_SUPPLY}")

In [ ]:
def find_attr(*needles):    for uid, name, _, _, _ in ATTRS:        if any(n.lower() in name.lower() for n in needles):            return uid    return NoneATTRIBUTE_MAP = {k: v for k, v in {    "first_name": find_attr("first name", "given"),    "last_name":  find_attr("last name", "surname", "family"),    "sex":        find_attr("gender", "sex"),}.items() if v}FIELDS = ["first_name", "last_name", "age", "sex", "facility",          "visit_date", "test_result", "treatment_given"]print("mapped to DHIS2 :", ATTRIBUTE_MAP)print("no mapping      :", [f for f in FIELDS                            if f not in ATTRIBUTE_MAP and f != "visit_date"])print("\nFields with no mapping are extracted and then NOT saved. On this demo")print("that is expected — there is no malaria tracker programme. On your own")print("server you would add attributes or programme-stage data elements.")

## 4. A synthetic formFictional patient. Generated here so the notebook runs with nothing to upload.To use your own, replace this cell with `files.upload()` and set `FORM`.

In [ ]:
from PIL import Image, ImageDraw, ImageFontimport matplotlib, osFD  = os.path.join(os.path.dirname(matplotlib.__file__), "mpl-data", "fonts", "ttf")BIG = ImageFont.truetype(os.path.join(FD, "DejaVuSans-Bold.ttf"), 30)REG = ImageFont.truetype(os.path.join(FD, "DejaVuSans.ttf"), 20)PEN = ImageFont.truetype(os.path.join(FD, "DejaVuSerif-Italic.ttf"), 25)INK, BLUE, GREY = (30, 32, 38), (24, 48, 130), (150, 152, 158)img = Image.new("RGB", (900, 1120), (252, 251, 246))d = ImageDraw.Draw(img)d.text((60, 52), "MINISTRY OF HEALTH AND SANITATION", font=REG, fill=INK)d.text((60, 86), "MALARIA CASE REPORT FORM", font=BIG, fill=INK)d.text((60, 130), "National Malaria Control Programme  ·  Form MCR-04",       font=REG, fill=GREY)d.line([60, 166, 840, 166], fill=INK, width=2)rows = [("Patient name", "Fatmata"), ("Surname", "Kargbo"),        ("Age (years)", "27"), ("Sex", "Female"),        ("Health facility", "Ngelehun CHC"), ("Date of visit", "12 March 2024"),        ("Malaria test result", "Positive"), ("Treatment given", "ACT")]y = 210for label, value in rows:    d.text((70, y), label, font=REG, fill=INK)    d.text((340, y - 4), value, font=PEN, fill=BLUE)    d.line([330, y + 32, 780, y + 32], fill=GREY, width=1)    y += 66d.text((60, 1050),       "SYNTHETIC TEST DOCUMENT — fictional patient, not a real medical record.",       font=REG, fill=(170, 60, 60))FORM = "form.png"img.save(FORM)print(f"{FORM} generated")img.resize((450, 560))

## 5. ExtractThe prompt does two things deliberately. It tells the model to return an emptystring when unsure — on a health form a blank is recoverable, a plausibleinvention is not. And it forbids `true`/`false`: a boolean leaking into a namefield produced a real record on this demo reading `Last name: "false"`. Theimport succeeded and the data was wrong, which is the worst combination there is.

In [ ]:
schema = ",\n".join(f'  "{f}": ""' for f in FIELDS)PROMPT = f'''You are transcribing a health facility form for data entry.Return ONLY a JSON object with exactly these keys:{{{schema}}}Rules:- Transcribe only what is legibly written or ticked. Do not infer or complete.- For checkboxes, return the label of the ticked option as a string.- Never return true or false. Every value is a string.- If a field is absent, illegible, or you are unsure, return an empty string.  An empty string is always better than a guess.- Preserve dates exactly as written on the form.- No commentary, no markdown fences.'''media = {"png": "image/png", "jpg": "image/jpeg",         "jpeg": "image/jpeg", "webp": "image/webp"}[FORM.rsplit(".", 1)[-1].lower()]b64 = base64.standard_b64encode(open(FORM, "rb").read()).decode()msg = client.messages.create(    model=MODEL, max_tokens=1024,    messages=[{"role": "user", "content": [        {"type": "image", "source": {"type": "base64",                                     "media_type": media, "data": b64}},        {"type": "text", "text": PROMPT}]}])raw = msg.content[0].texttry:    data = json.loads(raw[raw.find("{"):raw.rfind("}") + 1])except (ValueError, json.JSONDecodeError):    raise SystemExit(f"model did not return JSON. It said:\n{raw[:800]}")data = {f: data.get(f, "") for f in FIELDS}print(json.dumps(data, indent=2, ensure_ascii=False))print(f"\ntokens: {msg.usage.input_tokens} in, {msg.usage.output_tokens} out")

## 6. Validate and confirm`parse_date` **raises** rather than substituting a default. An earlier versionreturned a hardcoded `2024-01-01` when parsing failed, which puts a fabricateddate into a patient record where nobody will ever notice it.`%m/%d/%Y` is deliberately absent: `03/04/2026` is ambiguous, and silentlychoosing an interpretation is the same class of error.

In [ ]:
FORMATS = ("%Y-%m-%d", "%d %B %Y", "%d %b %Y", "%B %d, %Y",           "%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y")def parse_date(value):    if not value or not str(value).strip():        raise ValueError("no date on the form — cannot submit a record without one")    for fmt in FORMATS:        try:            parsed = datetime.strptime(str(value).strip(), fmt).date()        except ValueError:            continue        if parsed > date.today():            raise ValueError(f"date {parsed} is in the future")        return parsed.isoformat()    raise ValueError(f"could not read date {value!r}; accepted: {', '.join(FORMATS)}")def coerce(value, field=""):    if isinstance(value, bool):        raise ValueError(f"refusing to submit boolean {value!r} for {field!r}")    text = str(value).strip()    if not text:        raise ValueError(f"refusing to submit an empty value for {field!r}")    return textOCCURRED = parse_date(data.get("visit_date"))print("Extracted")print("-" * 62)for f in FIELDS:    mark = "->" if f in ATTRIBUTE_MAP else "  "    print(f" {mark} {f:<18} {data.get(f) or '— not found —'}")print("-" * 62)print(f"date {data.get('visit_date')!r} normalised to {OCCURRED}")print("\n'->' marks fields that will reach DHIS2. The rest are extracted but")print("have nowhere to go on this programme, and will NOT be saved.")missing = [f for f in FIELDS if not data.get(f)]if missing:    print(f"\nMISSING: {', '.join(missing)}")assert input("\nSubmit to DHIS2? [y/N]: ").strip().lower() in ("y", "yes"), "aborted"

## 7. Submit, then prove it**HTTP 200 does not mean the record was saved.** It means DHIS2 accepted theimport job. The job can still fail validation afterwards and create nothing.Code that reports success here is lying to whoever photographed the form.This cell follows the job to completion and then reads the record back. Itfilters on a unique marker rather than on recency, because the public demo iswritable by anyone — while testing this, the most recent record at Ngelehun CHCturned out to be somebody else's.

In [ ]:
MARKER = data["last_name"] + "-" + uuid.uuid4().hex[:6].upper()outgoing = dict(data, last_name=MARKER)attributes = [{"attribute": uid, "value": coerce(outgoing[f], f)}              for f, uid in ATTRIBUTE_MAP.items() if outgoing.get(f)]supplied = {a["attribute"] for a in attributes}missing_required = [u for u in YOU_MUST_SUPPLY if u not in supplied]assert not missing_required, f"required attributes missing: {missing_required}"payload = {"trackedEntities": [{    "trackedEntityType": TE_TYPE,    "orgUnit": ORG_UNIT,    "attributes": attributes,    "enrollments": [{"orgUnit": ORG_UNIT, "program": PROGRAM,                     "enrolledAt": OCCURRED, "occurredAt": OCCURRED,                     "status": "ACTIVE"}]}]}print(json.dumps(payload, indent=2))resp = s.post(f"{DHIS2}/api/tracker", json=payload, timeout=60)print(f"\nHTTP {resp.status_code}")body = resp.json()

In [ ]:
# DHIS2 answers in two shapes: an async job envelope, or a synchronous import# report. Indexing blindly into one crashes on the other.inner = body.get("response", body)if isinstance(inner, dict) and "id" in inner and "stats" not in inner:    job = inner["id"]    print(f"queued as job {job}, polling...")    for _ in range(6):        time.sleep(2)        rep = s.get(f"{DHIS2}/api/tracker/jobs/{job}/report", timeout=TIMEOUT)        if rep.status_code < 400:            inner = rep.json().get("response", rep.json())            if inner.get("status") not in (None, "PENDING", "RUNNING"):                breakstats = inner.get("stats", {})print(f"status  : {inner.get('status', body.get('status', 'UNKNOWN'))}")print(f"created : {stats.get('created', 0)}")print(f"ignored : {stats.get('ignored', 0)}")for key in ("errorReports", "warningReports"):    for e in (inner.get("validationReport") or {}).get(key) or []:        print(f"  {key}: {e.get('errorCode', '')} {e.get('message', '')}")

In [ ]:
# The only proof that matters.q = s.get(f"{DHIS2}/api/tracker/trackedEntities",          params={"orgUnit": ORG_UNIT, "program": PROGRAM, "ouMode": "SELECTED",                  "filter": f"{ATTRIBUTE_MAP['last_name']}:eq:{MARKER}",                  "fields": "trackedEntity,createdAt,"                            "attributes[displayName,value]"},          timeout=TIMEOUT).json()# 2.40 returns "instances"; later versions use "trackedEntities".found = q.get("instances") or q.get("trackedEntities") or []if found:    print(f"CONFIRMED — {len(found)} record read back from DHIS2\n")    for te in found:        print(f"trackedEntity {te['trackedEntity']}  created {te.get('createdAt','')}")        for a in te["attributes"]:            print(f"    {a.get('displayName', ''):<28} {a['value']}")    print(f"\nOpen it in the Capture app: {DHIS2}/dhis-web-capture/")else:    print("NOT CONFIRMED — the import reported success but no record came back.")    print("Check the validation messages above. This is exactly the failure that")    print("a status-code-only check would have reported as success.")

---## If something failed| Symptom | Cause ||---|---|| `Expecting value: line 1 column 1` | Server URL wrong — a retired host is serving an HTML 404 || `SecretNotFoundError` | Secret misnamed, or **Notebook access** toggle is off || HTTP 409 on submit | A required attribute is missing, or a value failed validation. The `errorReports` name the field || Model id rejected | Cell 2 lists what exists and picks automatically || Confirmed 0 records | Read the validation messages. The import was accepted and then discarded || Extraction wrong | Photograph quality dominates. Checkboxes are the weakest case — test them on your own forms |## What changed from the earlier version- Server URL: `play.dhis2.org/40` (retired, nginx 404) → `play.im.dhis2.org/stable-2-40-12`- Success now requires reading the record back, not an HTTP 200 and a job id- `parse_date` raises instead of substituting `2024-01-01`- Booleans are refused before submission- UIDs are discovered from the server, with `mandatory` and `generated` read- Unmapped fields are reported instead of silently dropped- Config key mismatch (`dhis2_program` vs `program`) removed with the single  configuration block- API key read from Secrets once, never assigned a literal string